In [11]:
import os
import pandas as pd
import pyodbc
from datetime import datetime
import warnings

# Configurações iniciais
usuario = os.getenv('USERNAME')
warnings.filterwarnings('ignore')

# Caminho e leitura das credenciais
path = fr'C:\Users\{usuario}\OneDrive - CAIXA Consórcio\Documentos\SENHA_BANCO_DADOS'
os.chdir(path)
df_senhas = pd.read_excel('SENHAS.xlsx')
server, database, username, password = df_senhas.iloc[0, 0:4]

# Conexão com o banco
conn = pyodbc.connect(
    f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={server};DATABASE={database};UID={username};PWD={password}'
)

# Consulta SQL
query = """
SELECT 
    FT.AnoMesRef, FT.ID_Produto, FT.ID_Comissionado, FT.ID_UF, FT.ID_Cota,
    FT.ST_Contrato, FT.Tipo_Pessoa, FT.Grupo, FT.Cota, FT.Faixa_Atraso,
    FT.Desclassificado, FT.TP_Pagto_DemaisParcelas, FT.ST_Adimplencia,
    DP.CD_InscricaoNacional, VA.DT_EntregaBem, CO.CANAL_VENDA, VB.DT_Contemplacao,
    CASE 
        WHEN VB.DT_Contemplacao IS NULL OR VA.DT_EntregaBem IS NULL THEN NULL
        WHEN DATEDIFF(DAY, VB.DT_Contemplacao, VA.DT_EntregaBem) < 31 THEN 'Over 30'
        WHEN DATEDIFF(DAY, VB.DT_Contemplacao, VA.DT_EntregaBem) < 61 THEN 'Over 60'
        WHEN DATEDIFF(DAY, VB.DT_Contemplacao, VA.DT_EntregaBem) < 91 THEN 'Over 90'
        WHEN DATEDIFF(DAY, VB.DT_Contemplacao, VA.DT_EntregaBem) < 121 THEN 'Over 120'
        WHEN DATEDIFF(DAY, VB.DT_Contemplacao, VA.DT_EntregaBem) < 151 THEN 'Over 150'
        WHEN DATEDIFF(DAY, VB.DT_Contemplacao, VA.DT_EntregaBem) < 181 THEN 'Over 180'
        WHEN DATEDIFF(DAY, VB.DT_Contemplacao, VA.DT_EntregaBem) < 211 THEN 'Over 210'
        WHEN DATEDIFF(DAY, VB.DT_Contemplacao, VA.DT_EntregaBem) < 241 THEN 'Over 240'
        WHEN DATEDIFF(DAY, VB.DT_Contemplacao, VA.DT_EntregaBem) < 271 THEN 'Over 270'
        WHEN DATEDIFF(DAY, VB.DT_Contemplacao, VA.DT_EntregaBem) < 301 THEN 'Over 300'
        WHEN DATEDIFF(DAY, VB.DT_Contemplacao, VA.DT_EntregaBem) < 331 THEN 'Over 330'
        WHEN DATEDIFF(DAY, VB.DT_Contemplacao, VA.DT_EntregaBem) < 361 THEN 'Over 360'
        WHEN DATEDIFF(DAY, VB.DT_Contemplacao, VA.DT_EntregaBem) < 391 THEN 'Over 390'
        ELSE 'Over > 390'
    END AS Categoria_Tempo_Entrega
FROM FT0015_CarteiraCotas AS FT
LEFT JOIN DM0013_Pessoas AS DP ON FT.ID_Pessoa = DP.ID_Pessoa
LEFT JOIN FT0002_VendaAlocacoes AS VA ON FT.ID_Cota = VA.ID_Cota
LEFT JOIN DM0004_Comissionados AS CO ON FT.ID_Comissionado = CO.ID_Comissionado
LEFT JOIN FT0018_Contemplacao AS VB ON FT.ID_Cota = VB.ID_Cota
WHERE FT.AnoMesRef > '202311'
"""

# Executa a query
print("📥 Executando a consulta SQL...")
df = pd.read_sql(query, conn)
df['DT_EntregaBem'] = pd.to_datetime(df['DT_EntregaBem'], errors='coerce')
df['DT_Contemplacao'] = pd.to_datetime(df['DT_Contemplacao'], errors='coerce')
df['Safra Contemplacao'] = df['DT_Contemplacao'].dt.strftime('%Y%m')
df['ID_Produto'] = df['ID_Produto'].astype(str)

# Mapeamento de categorias
mapa = {
    'Over 30': 'M1', 'Over 60': 'M2', 'Over 90': 'M3', 'Over 120': 'M4',
    'Over 150': 'M5', 'Over 180': 'M6', 'Over 210': 'M7', 'Over 240': 'M8',
    'Over 270': 'M9', 'Over 300': 'M10', 'Over 330': 'M11', 'Over 360': 'M12',
    'Over 390': 'M13', 'Over > 390': 'M14'
}
df['Categoria_M'] = df['Categoria_Tempo_Entrega'].map(mapa)
df_valid = df.dropna(subset=['Categoria_M', 'Safra Contemplacao'])
colunas_m = sorted(df_valid['Categoria_M'].unique(), key=lambda x: int(x[1:]))

# Função para criar matriz
def criar_matriz(df_base, coluna_chave):
    matriz = df_base.groupby(['Safra Contemplacao', 'Categoria_M'])[coluna_chave] \
        .nunique().unstack(fill_value=0).reset_index()
    matriz = matriz.reindex(columns=['Safra Contemplacao'] + colunas_m)
    matriz = matriz[matriz['Safra Contemplacao'] >= '202403']
    total = matriz[colunas_m].sum(axis=1)
    for col in colunas_m:
        matriz[f'{col}_%'] = (matriz[col] / total).fillna(0).round(4)
    return matriz

# Filtros
filtros = {
    'Geral Por Cotas Distintas': (None, 'ID_Cota'),
    'Geral Por Clientes': (None, 'CD_InscricaoNacional'),
    'Cotas Produto 107': (df_valid[df_valid['ID_Produto'] == '107'], 'ID_Cota'),
    'Clientes Produto 107': (df_valid[df_valid['ID_Produto'] == '107'], 'CD_InscricaoNacional'),
    'Cotas Produto 102': (df_valid[df_valid['ID_Produto'] == '102'], 'ID_Cota'),
    'Clientes Produto 102': (df_valid[df_valid['ID_Produto'] == '102'], 'CD_InscricaoNacional'),
    'Cotas Produto 103': (df_valid[df_valid['ID_Produto'] == '103'], 'ID_Cota'),
    'Clientes Produto 103': (df_valid[df_valid['ID_Produto'] == '103'], 'CD_InscricaoNacional'),
    'Cotas Pessoa Física': (df_valid[df_valid['Tipo_Pessoa'] == 'F'], 'ID_Cota'),
    'Clientes Pessoa Física': (df_valid[df_valid['Tipo_Pessoa'] == 'F'], 'CD_InscricaoNacional'),
    'Cotas Pessoa Jurídica': (df_valid[df_valid['Tipo_Pessoa'] == 'J'], 'ID_Cota'),
    'Clientes Pessoa Jurídica': (df_valid[df_valid['Tipo_Pessoa'] == 'J'], 'CD_InscricaoNacional'),
}

# Verificações de integridade
print("\n🔍 Verificando integridade dos dados...")

total_geral = df_valid['ID_Cota'].nunique()
total_pf = df_valid[df_valid['Tipo_Pessoa'] == 'F']['ID_Cota'].nunique()
total_pj = df_valid[df_valid['Tipo_Pessoa'] == 'J']['ID_Cota'].nunique()
total_outros = df_valid[~df_valid['Tipo_Pessoa'].isin(['F', 'J'])]['ID_Cota'].nunique()

duplicadas = df_valid[['ID_Cota', 'Tipo_Pessoa']].dropna().drop_duplicates()
duplicadas_count = duplicadas.groupby('ID_Cota')['Tipo_Pessoa'].nunique()
cotas_multiplos_tipos = duplicadas_count[duplicadas_count > 1].count()

print(f"🔢 Total Geral de Cotas Únicas: {total_geral}")
print(f"🧍 Pessoa Física (F): {total_pf}")
print(f"🏢 Pessoa Jurídica (J): {total_pj}")
print(f"❓ Tipo_Pessoa nulo ou inválido: {total_outros}")
print(f"🔄 Cotas com múltiplos tipos de pessoa (F + J): {cotas_multiplos_tipos}")
print(f"📊 Soma PF + PJ: {total_pf + total_pj}")

# Caminho de saída
excel_path = rf'C:\Users\{usuario}\OneDrive - CAIXA Consórcio\Documentos\TABELA Milton\matriz_tempo_entrega.xlsx'

# Exporta para Excel
with pd.ExcelWriter(excel_path, engine='xlsxwriter') as writer:
    for nome_planilha, (filtro_df, coluna) in filtros.items():
        base = df_valid if filtro_df is None else filtro_df
        matriz = criar_matriz(base, coluna)
        matriz.to_excel(writer, sheet_name=nome_planilha, index=False)

        # Formatação
        worksheet = writer.sheets[nome_planilha]
        for idx, col in enumerate(matriz.columns):
            largura = max(matriz[col].astype(str).map(len).max(), len(col)) + 1
            worksheet.set_column(idx, idx, largura)

        for idx, col in enumerate(colunas_m):
            if col in matriz.columns:
                worksheet.conditional_format(1, idx, len(matriz), idx, {
                    'type': '3_color_scale',
                    'min_color': "#63BE7B",
                    'mid_color': "#FFEB84",
                    'max_color': "#F8696B"
                })

print(f"\n✅ Excel exportado com sucesso:\n{excel_path}")


📥 Executando a consulta SQL...

🔍 Verificando integridade dos dados...
🔢 Total Geral de Cotas Únicas: 31525
🧍 Pessoa Física (F): 22774
🏢 Pessoa Jurídica (J): 9297
❓ Tipo_Pessoa nulo ou inválido: 0
🔄 Cotas com múltiplos tipos de pessoa (F + J): 546
📊 Soma PF + PJ: 32071

✅ Excel exportado com sucesso:
C:\Users\GabrielHenriqueSilva\OneDrive - CAIXA Consórcio\Documentos\TABELA Milton\matriz_tempo_entrega.xlsx
